<a href="https://colab.research.google.com/github/Julian6262/the_founder/blob/main/%D0%94%D0%97-29.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В домашней работе вам необходимо получить навыки работы с большими языковыми моделями. Для этого необходимо:

1. Создать учетную запись на Hugging Face (HF).
2. Получить токен для работы с HF.
3. Загрузить адаптер русско-язычной [Saiga](https://huggingface.co/IlyaGusev/saiga_mistral_7b_lora)
4. Для получению 3 баллов вам достаточно сгенерировать 3 шутки с помощью Saiga.
5. Для 4-х баллов выполните пункт 4, используя квантованную версию модели [saiga_mistral_7b_gguf](https://huggingface.co/IlyaGusev/saiga_mistral_7b_gguf).
6. Хотите 5 баллов? Используя датасет [ru_turbo_saiga](https://huggingface.co/datasets/IlyaGusev/ru_turbo_saiga), на котором обучалась данная модель произведите оценку точности, как мы это делали в практической части урока. Хотя на данном датасете оценивать модель не совсем корректно, так как она уже "видела" эти данные, но поняв принцип, вы всегда сможете повторить процедуру и на новых данных. Также вы можете оценить модель и другими способами, например, изложенными в [статье](https://www.philschmid.de/evaluate-llm).

In [ ]:
!pip install torch
!pip install transformers datasets accelerate evaluate bitsandbytes trl peft
!pip install ctransformers[cuda]

In [ ]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline

In [ ]:
# Базовая модель
base_model_name = "Open-Orca/Mistral-7B-OpenOrca"

# Идентификатор адаптера
model_id = "IlyaGusev/saiga_mistral_7b_lora"

In [ ]:
# Загрузка модели с PEFT адаптером
model = AutoPeftModelForCausalLM.from_pretrained(
    model_id,      # адаптер
    device_map={"": 0}, # использовать GPU:0, трюк если auto приводит к ошибке
    torch_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(
    base_model_name,          # базовая модель
    trust_remote_code=True)

tokenizer.pad_token = tokenizer.eos_token # определяем токен разделитель

# загрузка в pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
def generate_conversation(system_message, question, pipe):
    data = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": question},
    ]
    prompt = pipe.tokenizer.apply_chat_template(data, tokenize=False, add_generation_prompt=True)
    outputs = pipe(
    prompt,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.1,
    top_p=0.95,
    # eos_token_id=pipe.tokenizer.eos_token_id,
    # pad_token_id=pipe.tokenizer.pad_token_id
)

    print("Запрос:")
    print(data[1]['content'])
    print("\nСгенерированный ответ модели:")
    print(outputs[0]['generated_text'][len(prompt):].strip())

In [ ]:
system_message = 'Ты врач Гинеколог. Тебе задают вопросы пациенты, отвечай на вопросы в соответствии с темой "Планирование беременности"'
question = 'При планировании беременности помогает Фемибион 1 забеременеть???'
generate_conversation(system_message, question, pipe)

Запрос:
При планировании беременности помогает Фемибион 1 забеременеть???

Сгенерированный ответ модели:
Фемибион 1 - это препарат, который помогает женщинам забеременеть. Он содержит гормоны, которые стимулируют выработку яичников и увеличивают их функциональность, что способствует оплодотворению. Однако, прежде чем приступить к применению Фемибиона 1, рекомендуется консультироваться с врачом-гинекологом, который будет


In [ ]:
system_message = 'Ты рецептурный мастер. Твоя задача создавать и оптимизировать кулинарные рецепты'
question = 'Создай рецепт супа с курицей и грибами'
generate_conversation(system_message, question, pipe)

Запрос:
Создай рецепт супа с курицей и грибами

Сгенерированный ответ модели:
Рецепт супа с курицей и грибами:

Ингредиенты:
- 1 кг курицы
- 1 кг грибов (шампиньоны, шеллы, остроголовые)
- 1 литр воды
- 1 столовая ложка соли
- 1 столовая ложка петрушки
- 1 столовая ложка зеленого лука
- 1 столовая ло


In [ ]:
system_message = 'Ты автомеханик. Твоя задача помогать советами с ремонтом автомобиля'
question = 'как называется агрегат, который приводит в движение автомобиль?'
generate_conversation(system_message, question, pipe)

Запрос:
как называется агрегат, который приводит в движение автомобиль?

Сгенерированный ответ модели:
Агрегат, который приводит в движение автомобиль, называется двигателем внутреннего сгорания (ДВС). Он преобразует энергию, полученную от сгорания топлива (бензина или дизельного топлива) с помощью поршней и цилиндров, в механическую энергию, которая вращает коленчатый вал и, в конечном итоге,


In [ ]:
del model, pipe, tokenizer

In [ ]:
torch.cuda.empty_cache()

### **saiga_mistral_7b_gguf**

In [ ]:
from ctransformers import AutoModelForCausalLM

# Загружаем LLM и Tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "IlyaGusev/saiga_mistral_7b_gguf",
    model_type="mistral",                       # уточняем тип модели (на базе Mistral AI)
    gpu_layers=50,                              # сколько слоев загружать в GPU
    hf=True                                     # создавать модель-трансформер с помощью HuggingFace.
)

tokenizer = AutoTokenizer.from_pretrained(
    base_model_name, use_fast=True
)

# Create a pipeline
pipe = pipeline(model=model, tokenizer=tokenizer, task='text-generation')

In [ ]:
system_message = 'Ты врач Гинеколог. Тебе задают вопросы пациенты, отвечай на вопросы в соответствии с темой "Планирование беременности"'
question = 'При планировании беременности помогает Фемибион 1 забеременеть???'
generate_conversation(system_message, question, pipe)

Запрос:
При планировании беременности помогает Фемибион 1 забеременеть???

Сгенерированный ответ модели:
Фемибион 1 - это препарат, который содержит гормоны, которые стимулируют выработку яичных клеток и увеличение количества яичных клеток, что может способствовать забеременению. Однако, это не гарантирует забеременение, так как забеременение зависит от многих факторов, включая возраст, здоровье, у


In [ ]:
system_message = 'Ты рецептурный мастер. Твоя задача создавать и оптимизировать кулинарные рецепты'
question = 'Создай рецепт супа с курицей и грибами'
generate_conversation(system_message, question, pipe)

Запрос:
Создай рецепт супа с курицей и грибами

Сгенерированный ответ модели:
Ингредиенты:

- 1 кг курицы
- 1 бут грибов (например, шампиньон)
- 1 бут овощей (например, картофель, морковь, лук)
- 1 стакан сметаны
- 1 стакан свежих зеленых трав (например, петрушка, укроль)
- 1 стакан бульона
- 1 стакан воды


In [ ]:
system_message = 'Ты автомеханик. Твоя задача помогать советами с ремонтом автомобиля'
question = 'как называется агрегат, который приводит в движение автомобиль?'
generate_conversation(system_message, question, pipe)

Запрос:
как называется агрегат, который приводит в движение автомобиль?

Сгенерированный ответ модели:
Агрегат, который приводит в движение автомобиль, называется двигатель (или двигатель внутреннего сгорания).
